# Ingest current AEMO ST PASA forecasts

Download current forecast runs and extract their native 30-minute CSVs into Bronze. The Silver merge reads current_uncompressed alongside the archived forecasts. This notebook can run independently of archive ingestion.

## 1. Discover current runs

Match PUBLIC_STPASA_YYYYMMDDHHMM_<numeric run ID>.zip in the current listing.

In [ ]:
import os
import re
import time
import zipfile
from urllib.parse import urljoin

import requests


current_url = (
    "https://www.nemweb.com.au/"
    "Reports/CURRENT/Short_Term_PASA_Reports/"
)

def discover_zip_urls(folder, filename_pattern):
    # Read the live directory listing rather than constructing filenames.
    response = requests.get(folder, timeout=60)
    response.raise_for_status()

    files = re.findall(filename_pattern, response.text)

    return [
        urljoin(folder, filename)
        for filename in sorted(set(files))
    ]


current_urls = discover_zip_urls(
    current_url,
    r"PUBLIC_STPASA_\d{12}_\d+\.zip",
)


In [ ]:
print(f"Found {len(current_urls)} current forecast runs")
current_urls

## 2. Download missing current ZIPs

Refresh the live listing before downloading. Skip existing files and report expired files returning 404; other HTTP errors stop the run.

In [ ]:
def download_if_not_exists(url, bronze_folder, skip_expired=False):
    filename = os.path.basename(url)
    path = os.path.join(bronze_folder, filename)

    if os.path.exists(path):
        print(f"Skipping: {filename}")
        return path

    print(f"Downloading: {filename}")
    start = time.time()

    response = requests.get(url, timeout=60)

    # Files can leave NEMWeb's live Current directory after discovery.
    if skip_expired and response.status_code == 404:
        print(f"Expired before download: {filename}")
        return None

    response.raise_for_status()

    with open(path, "wb") as file:
        file.write(response.content)

    print(f"Finished: {filename} - {time.time() - start:.1f} seconds")

    return path

In [ ]:
current_folder = (
    "/Volumes/workspace/default/aemo_mlops_volume/"
    "bronze/forecast_30min/current"
)
os.makedirs(current_folder, exist_ok=True)

# Refresh the volatile listing immediately before downloading.
current_urls = discover_zip_urls(current_url, r"PUBLIC_STPASA_\d{12}_\d+\.zip")
for url in current_urls:
    download_if_not_exists(url, current_folder, skip_expired=True)


## 3. Extract current forecasts

Each current ZIP contains a CSV directly. Extract into current_uncompressed and skip already-extracted runs.

In [ ]:
current_csv_folder = current_folder + "_uncompressed"

os.makedirs(current_csv_folder, exist_ok=True)

current_zip_files = [
    filename
    for filename in os.listdir(current_folder)
    if filename.endswith(".zip")
]

start = time.time()
print(f"Found {len(current_zip_files)} current ZIPs")

for i, filename in enumerate(sorted(current_zip_files), 1):
    path = os.path.join(current_folder, filename)

    with zipfile.ZipFile(path, "r") as zip_file:
        members = zip_file.namelist()

        already_extracted = all(
            os.path.exists(
                os.path.join(current_csv_folder, os.path.basename(member))
            )
            for member in members
        )

        if already_extracted:
            print(f"[{i}/{len(current_zip_files)}] Skipping: {filename}")
            continue

        print(f"[{i}/{len(current_zip_files)}] Extracting: {filename}")
        zip_file.extractall(current_csv_folder)

print(f"Finished current extraction in {time.time() - start:.1f} seconds")